# 5. The scorecard

The model is now written as a scorecard. Each bin of each feature gets a number of points, and a borrower's score is the sum of their points. A higher score means a safer borrower. The scorecard gives exactly the same ranking as the model. It is only an easier way to read it.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from optbinning import BinningProcess

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

binning_36 = BinningProcess.load(str(PROCESSED / "binning_36.pkl"))
binning_60 = BinningProcess.load(str(PROCESSED / "binning_60.pkl"))

weights_36 = pd.read_json(PROCESSED / "weights_36.json", typ="series")
weights_60 = pd.read_json(PROCESSED / "weights_60.json", typ="series")

### 5.1 The scale

The model works in log odds, which are hard to read. A scorecard stretches them onto a scale of points that is fixed by two choices.

- A score of 600 means odds of 50 to 1, that is 50 borrowers who repay for each one who defaults.
- Every 20 points more doubles the odds. A score of 620 means 100 to 1, and 580 means 25 to 1.

In [2]:
PDO = 20            # points to double the odds
BASE_SCORE = 600
BASE_ODDS = 50

FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)

print(f"Factor: {FACTOR:.2f}")
print(f"Offset: {OFFSET:.2f}")

Factor: 28.85
Offset: 487.12


### 5.2 Points for each bin

Every borrower starts with the same base points, which come from the constant of the model. Each bin then adds or takes away points. A bin's points are its WoE times the weight of its feature, times the factor, with the sign changed so that safer bins get more points.

In [3]:
def scorecard(binning, weights):
    rows = []
    for feature in weights.index.drop("const"):
        table = binning.get_binned_variable(feature).binning_table.build()
        table = table[(table.index != "Totals") & (table["Count"] > 0)]
        for _, row in table.iterrows():
            name = row["Bin"] if isinstance(row["Bin"], str) else ", ".join(row["Bin"])
            rows.append({
                "feature": feature,
                "bin": name,
                "loans": row["Count"],
                "woe": round(float(row["WoE"]), 3),
                "points": round(-FACTOR * weights[feature] * float(row["WoE"])),
            })
    return pd.DataFrame(rows)
    

In [4]:
base_36 = round(OFFSET - FACTOR * weights_36["const"])
card_36 = scorecard(binning_36, weights_36)

print(f"Base points: {base_36}")
card_36

Base points: 542


,feature,bin,loans,woe,points
0,purpose,"car, major_purchase, wedding, credit_card",93051,0.195,5
1,purpose,home_improvement,18487,0.163,4
2,purpose,"debt_consolidation, house, vacation",193641,-0.048,-1
3,purpose,"other, educational, medical, renewable_energy,...",30379,-0.302,-8
4,home_ownership,"ANY, MORTGAGE",157513,0.212,3
...,...,...,...,...,...
68,open_acc,"(-inf, 6.50)",56691,0.020,2
69,open_acc,"[6.50, 8.50)",59317,0.015,2
70,open_acc,"[8.50, 13.50)",135191,-0.002,0
71,open_acc,"[13.50, 19.50)",64669,-0.008,-1


In [5]:
card_36[card_36["feature"] == "fico"]

,feature,bin,loans,woe,points
43,fico,"(-inf, 664.50)",28076,-0.417,-10
44,fico,"[664.50, 669.50)",28307,-0.346,-9
45,fico,"[669.50, 674.50)",28744,-0.318,-8
46,fico,"[674.50, 679.50)",26130,-0.211,-5
47,fico,"[679.50, 684.50)",26157,-0.193,-5
48,fico,"[684.50, 689.50)",23572,-0.116,-3
49,fico,"[689.50, 694.50)",22721,-0.016,0
50,fico,"[694.50, 699.50)",20632,0.013,0
51,fico,"[699.50, 709.50)",35136,0.145,4
52,fico,"[709.50, 719.50)",26746,0.310,8


In [6]:
card_36[card_36["feature"] == "purpose"]

,feature,bin,loans,woe,points
0,purpose,"car, major_purchase, wedding, credit_card",93051,0.195,5
1,purpose,home_improvement,18487,0.163,4
2,purpose,"debt_consolidation, house, vacation",193641,-0.048,-1
3,purpose,"other, educational, medical, renewable_energy,...",30379,-0.302,-8


### 5.3 The 60-month scorecard

The same is done for the 60-month model. Its base points are lower, because these loans default more often.

In [7]:
base_60 = round(OFFSET - FACTOR * weights_60["const"])
card_60 = scorecard(binning_60, weights_60)

print(f"Base points: {base_60}")
card_60

Base points: 518


,feature,bin,loans,woe,points
0,inq_last_6mths,"(-inf, 0.50)",15061,0.158,5
1,inq_last_6mths,"[0.50, 1.50)",9454,-0.082,-3
2,inq_last_6mths,"[1.50, 2.50)",5054,-0.130,-4
3,inq_last_6mths,"[2.50, inf)",3268,-0.238,-7
4,fico,"(-inf, 669.50)",3478,-0.428,-11
5,fico,"[669.50, 674.50)",2127,-0.377,-10
6,fico,"[674.50, 684.50)",4446,-0.260,-7
7,fico,"[684.50, 689.50)",2267,-0.128,-3
8,fico,"[689.50, 694.50)",2212,-0.125,-3
9,fico,"[694.50, 699.50)",2196,-0.019,0


### 5.4 Scoring the test loans

Each test loan is scored by adding up its points. The table shows, for the first loans, the base points, the points from each feature, and the total score.

In [8]:
test_36 = pd.read_parquet(PROCESSED / "test_36.parquet")
test_60 = pd.read_parquet(PROCESSED / "test_60.parquet")
features = [c for c in test_36.columns if c != "target"]


def score_loans(loans, binning, weights):
    woe = binning.transform(loans[features], metric="woe")
    chosen = weights.index.drop("const")
    points = (-FACTOR * woe[chosen] * weights[chosen]).round().astype(int)
    points.insert(0, "base", round(OFFSET - FACTOR * weights["const"]))
    points["score"] = points.sum(axis=1)
    return points


points_36 = score_loans(test_36, binning_36, weights_36)
points_60 = score_loans(test_60, binning_60, weights_60)

points_36.head()

,base,purpose,home_ownership,annual_inc,dti,inq_last_6mths,total_acc,fico,loan_to_income,verification_status,open_acc,score
0,542,-1,3,0,5,-2,-3,-5,6,2,2,549
1,542,-8,3,1,1,-10,3,8,-8,2,-6,528
2,542,-1,-3,-5,4,5,-6,0,-8,-1,2,529
3,542,-1,3,7,2,5,2,-5,4,2,0,561
4,542,5,3,4,0,5,-2,4,-1,2,2,564


### 5.5 Do the scores work?

Two checks. First, the scores should rank borrowers as well as the PDs did, so the Gini should match the one from notebook 4. Second, the default rate should fall as the score rises. The loans are grouped into bands of 20 points to show this.

In [12]:
from sklearn.metrics import roc_auc_score

for name, points, test in [("36-month", points_36, test_36), ("60-month", points_60, test_60)]:
    auc = roc_auc_score(test["target"], -points["score"])
    print(f"{name}: AUC {auc:.3f}, Gini {2 * auc - 1:.3f}")

36-month: AUC 0.654, Gini 0.308
60-month: AUC 0.607, Gini 0.213


The Gini is 0.308 for the 36-month loans and 0.213 for the 60-month loans, against 0.307 and 0.214 from the PDs in notebook 4. The tiny differences come from rounding the points to whole numbers. Therefore the scorecard ranks borrowers exactly as well as the model, as it should.

The second check is whether the scores mean what they should. The test loans are grouped into bands of 20 points. For each band, the table shows how many loans it holds and what share of them defaulted. The default rate should fall as the score rises.

In [13]:
def score_bands(score, y, width=20):
    low = (score // width) * width
    bands = low.astype(str) + " to " + (low + width - 1).astype(str)
    table = pd.DataFrame({"band": bands, "low": low, "target": y}).groupby(["low", "band"]).agg(
        loans=("target", "count"), default_rate=("target", "mean")).reset_index(level="low", drop=True)
    table.insert(1, "share_of_loans", table["loans"] / table["loans"].sum())
    return table.round(3)


score_bands(points_36["score"], test_36["target"])

,loans,share_of_loans,default_rate
band,,,
480 to 499,238,0.001,0.361
500 to 519,16989,0.060,0.292
520 to 539,95656,0.338,0.205
540 to 559,115569,0.408,0.124
560 to 579,45019,0.159,0.064
580 to 599,9264,0.033,0.025
600 to 619,291,0.001,0.014


In [14]:
score_bands(points_60["score"], test_60["target"])

,loans,share_of_loans,default_rate
band,,,
480 to 499,1557,0.070,0.376
500 to 519,11435,0.517,0.283
520 to 539,8128,0.367,0.191
540 to 559,995,0.045,0.107
560 to 579,17,0.001,0.059


In both groups, the default rate falls in every step as the score rises. For the 36-month loans it goes from 36.1% in the lowest band to 1.4% in the highest, and for the 60-month loans from 37.6% to 5.9%.

A given score also means about the same risk for both loan terms. Loans scoring 520 to 539 default 20.5% of the time in the 36-month group and 19.1% in the 60-month group. The two groups differ in where their borrowers sit. Most 36-month loans score between 520 and 559, and most 60-month loans between 500 and 539, which is why the 60-month loans default more overall.

The bands at the two ends hold few loans, so their default rates are less certain.

### 5.6 Explaining a decision

A lender should be able to tell an applicant why they got their score. The scorecard makes this simple. For one borrower, the table shows every feature with the points they got, the most points that feature can give, and the gap between the two.

The features with the largest gaps are the main reasons the score is not higher. The features with no gap are the borrower's strengths.

In [23]:
def explain(points, loans, card, i):
    got = points.loc[i].drop(["base", "score"])
    best = card.groupby("feature")["points"].max()[got.index]
    table = pd.DataFrame({
        "value": loans.loc[i, got.index],
        "points": got,
        "best_possible": best,
        "gap": got - best,
    })
    return table.sort_values("gap")

#### A 36-month borrower

In [ ]:
borrower = 2

print(f"Base points:          {points_36.loc[borrower, 'base']}")
print(f"Points from features: {points_36.loc[borrower].drop(['base', 'score']).sum():+d}")
print(f"Score:                {points_36.loc[borrower, 'score']}")

explain(points_36, test_36, card_36, borrower)

Base points:          542
Points from features: -13
Score:                529


,value,points,best_possible,gap
fico,692.0,0,28,-28
loan_to_income,0.351471,-8,6,-14
annual_inc,34000.0,-5,7,-12
total_acc,6.0,-6,3,-9
purpose,debt_consolidation,-1,5,-6
home_ownership,RENT,-3,3,-6
verification_status,Source Verified,-1,2,-3
dti,10.2,4,6,-2
inq_last_6mths,0.0,5,5,0
open_acc,5.0,2,2,0


#### A 60-month borrower

In [22]:
borrower = 2

print(f"Base points:          {points_60.loc[borrower, 'base']}")
print(f"Points from features: {points_60.loc[borrower].drop(['base', 'score']).sum():+d}")
print(f"Score:                {points_60.loc[borrower, 'score']}")

explain(points_60, test_60, card_60, borrower)

Base points:          518
Points from features: +5
Score:                523


,value,points,best_possible,gap
fico,677.0,-7,22,-29
home_ownership,RENT,-3,2,-5
loan_to_income,0.09841,8,11,-3
annual_inc,105680.0,2,5,-3
inq_last_6mths,0.0,5,5,0
pub_rec,0.0,0,0,0


### 5.7 Saving the scorecards

The two scorecards are saved as tables in the `reports` folder, so they can be read without running any code.

In [25]:
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

card_36.to_csv(REPORTS / "scorecard_36.csv", index=False)
card_60.to_csv(REPORTS / "scorecard_60.csv", index=False)

print(f"36-month: base {base_36} points, {len(card_36)} bins")
print(f"60-month: base {base_60} points, {len(card_60)} bins")

36-month: base 542 points, 73 bins
60-month: base 518 points, 39 bins
